# 00 Architecture

Model summary, module tree and tensor shapes at each stage, for the specified model and for the CPU fallback model. Weights are not downloaded here.

In [1]:
import json, os, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd
from IPython.display import Image, display
pd.set_option("display.max_columns", 40, "display.width", 200, "display.precision", 4)
RESULTS, CHECKPOINTS = ROOT / "results", ROOT / "checkpoints"

def require(*paths, hint):
    missing = [str(p) for p in paths if not Path(p).exists()]
    if missing:
        print("MISSING:", *missing, sep="\n  ")
        print("To create:", hint)
    return not missing

def runs():
    return sorted(p for p in CHECKPOINTS.iterdir() if (p / "run.json").is_file()) if CHECKPOINTS.is_dir() else []


In [2]:
import dataclasses, torch
from torchinfo import summary
from xaidr.config import load_config
from xaidr.models.dual_head import DRDualHeadModel

def describe(config_file):
    cfg = load_config(ROOT / "configs" / config_file)
    size = cfg.data.image_size
    model = DRDualHeadModel(dataclasses.replace(cfg.model, pretrained=False), size).eval()
    with torch.no_grad():
        out = model(torch.zeros(1, 3, size, size))
    print(f"{cfg.model.backbone}, input 1 x 3 x {size} x {size}")
    for key, value in out.items():
        print(f"  {key:13s} {tuple(value.shape)}")
    return summary(model, input_size=(1, 3, size, size), depth=3, verbose=0,
                   col_names=("output_size", "num_params"))

describe("default.yaml")

C:\Users\abhir\Downloads\MJ\github\Retinopathy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


maxvit_tiny_tf_512, input 1 x 3 x 512 x 512
  det_logit     (1,)
  grade_logits  (1, 5)
  pooled        (1, 512)
  feature_map   (1, 512, 16, 16)


Layer (type:depth-idx)                                                 Output Shape              Param #
DRDualHeadModel                                                        [1, 512, 16, 16]          --
├─Backbone: 1-1                                                        --                        --
│    └─MaxxVit: 2-1                                                    --                        --
│    │    └─Stem: 3-1                                                  [1, 64, 256, 256]         38,848
│    │    └─Sequential: 3-2                                            [1, 512, 16, 16]          30,234,056
│    │    └─Identity: 3-3                                              [1, 512, 16, 16]          --
│    │    └─NormMlpClassifierHead: 3-4                                 [1, 512]                  263,680
├─DetectionHead: 1-2                                                   [1]                       --
│    └─Sequential: 2-2                                                 [1, 1] 

In [3]:
describe("cpu_fallback.yaml")

convnext_tiny, input 1 x 3 x 256 x 256
  det_logit     (1,)
  grade_logits  (1, 5)
  pooled        (1, 768)
  feature_map   (1, 768, 8, 8)


Layer (type:depth-idx)                                       Output Shape              Param #
DRDualHeadModel                                              [1, 768, 8, 8]            --
├─Backbone: 1-1                                              --                        --
│    └─ConvNeXt: 2-1                                         --                        --
│    │    └─Sequential: 3-1                                  [1, 96, 64, 64]           4,896
│    │    └─Sequential: 3-2                                  [1, 768, 8, 8]            27,813,696
│    │    └─Identity: 3-3                                    [1, 768, 8, 8]            --
│    │    └─NormMlpClassifierHead: 3-4                       [1, 768]                  1,536
├─DetectionHead: 1-2                                         [1]                       --
│    └─Sequential: 2-2                                       [1, 1]                    --
│    │    └─Linear: 3-5                                      [1, 256]            

```
image 3 x S x S -> shared backbone -> feature map C x h x w -> pooled features
                                   |-> detection head (Linear-ReLU-Dropout-Linear) -> 1 logit
                                   |-> grading head   (Linear-ReLU-Dropout-Linear) -> 5 logits
loss = 0.4 * BCE(detection) + 0.6 * class-weighted CE(grading)
```